In [577]:
%pip install pyopengl --break-system-packages
%pip install glfw --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [578]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import math

In [579]:
vertex_shader_source = """
#version 330 core
layout (location = 0) in vec3 position;
uniform mat4 model;

void main() {
    // Aplica a matriz de transformação do objeto
    gl_Position = model * vec4(position, 1.0);
}
"""

fragment_shader_source = """
#version 330 core
out vec4 FragColor;
uniform vec4 color;

void main() {
    // Aplica a cor sólida ao objeto (sem iluminação/textura)
    FragColor = color;
}
"""

In [580]:
def mat_identity():
    return np.identity(4, dtype=np.float32).T

def mat_translate(tx, ty, tz):
    return np.array([
        [1.0, 0.0, 0.0, tx],
        [0.0, 1.0, 0.0, ty],
        [0.0, 0.0, 1.0, tz],
        [0.0, 0.0, 0.0, 1.0]
    ], dtype=np.float32).T 

def mat_scale(sx, sy, sz):
    return np.array([
        [sx,  0.0, 0.0, 0.0],
        [0.0, sy,  0.0, 0.0],
        [0.0, 0.0, sz,  0.0],
        [0.0, 0.0, 0.0, 1.0]
    ], dtype=np.float32).T

def mat_rotate_y(angle_rad):
    c = math.cos(angle_rad)
    s = math.sin(angle_rad)
    return np.array([
        [ c,   0.0,  s,   0.0],
        [ 0.0, 1.0,  0.0, 0.0],
        [-s,   0.0,  c,   0.0],
        [ 0.0, 0.0,  0.0, 1.0]
    ], dtype=np.float32).T

def mat_rotate_z(angle_rad):
    c = math.cos(angle_rad)
    s = math.sin(angle_rad)
    return np.array([
        [ c,  -s,  0.0, 0.0],
        [ s,   c,  0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ], dtype=np.float32).T

In [581]:
wireframe_mode = False
barco_pos_x = 0.0
barco_pos_y = 0.0
peixe_escala = 1.0
vara_angulo = 0.0

In [582]:
pessoinha_angulo_y = 0.0 # Nova variável

def key_callback(window, key, scancode, action, mods):
    global wireframe_mode, barco_pos_x, barco_pos_y, pessoinha_angulo_y
    
    if action == glfw.PRESS or action == glfw.REPEAT:
        if key == glfw.KEY_P and action == glfw.PRESS:
            wireframe_mode = not wireframe_mode
            glPolygonMode(GL_FRONT_AND_BACK, GL_LINE if wireframe_mode else GL_FILL)
            
        # Translação do Barco (Setas) [cite: 17]
        elif key == glfw.KEY_RIGHT: barco_pos_x += 0.05
        elif key == glfw.KEY_LEFT:  barco_pos_x -= 0.05
        elif key == glfw.KEY_UP:    barco_pos_y += 0.05
        elif key == glfw.KEY_DOWN:  barco_pos_y -= 0.05
        
        # Rotação da Pessoinha (Q e R) 
        elif key == glfw.KEY_Q:     pessoinha_angulo_y += 0.1
        elif key == glfw.KEY_R:     pessoinha_angulo_y -= 0.1

In [583]:
def criar_lago(linhas=10, colunas=20):
    # O lago vai de x=-1.0 a x=1.0, e de y=-1.0 a y=0.0 (metade de baixo da tela)
    x_min, x_max = -1.0, 1.0
    y_min, y_max = -1.0, 0.0
    
    dx = (x_max - x_min) / colunas
    dy = (y_max - y_min) / linhas
    
    vertices = []
    
    for i in range(linhas):
        for j in range(colunas):
            x0 = x_min + j * dx
            y0 = y_min + i * dy
            x1 = x0 + dx
            y1 = y0 + dy
            z = 0.0 # Lago é 2D no plano Z=0
            
            # Primeiro triângulo do "quadradinho"
            vertices.extend([x0, y0, z])
            vertices.extend([x1, y0, z])
            vertices.extend([x0, y1, z])
            
            # Segundo triângulo do "quadradinho"
            vertices.extend([x1, y0, z])
            vertices.extend([x1, y1, z])
            vertices.extend([x0, y1, z])
            
    return np.array(vertices, dtype=np.float32)

In [584]:
def criar_barco():
    # Definindo os 8 pontos (vértices) principais do casco do barco
    # Formato: [x, y, z]
    b1 = [-0.4, -0.2,  0.2] # Base frente-esquerda
    b2 = [ 0.4, -0.2,  0.2] # Base frente-direita
    b3 = [ 0.4, -0.2, -0.2] # Base trás-direita
    b4 = [-0.4, -0.2, -0.2] # Base trás-esquerda

    t1 = [-0.6,  0.2,  0.3] # Topo frente-esquerda
    t2 = [ 0.6,  0.2,  0.3] # Topo frente-direita
    t3 = [ 0.6,  0.2, -0.3] # Topo trás-direita
    t4 = [-0.6,  0.2, -0.3] # Topo trás-esquerda

    # Compondo o objeto com triângulos (2 triângulos por face)
    faces = [
        # Fundo do barco
        b1, b2, b3,    b1, b3, b4,
        # Topo (Convés)
        t1, t3, t2,    t1, t4, t3,
        # Lateral Frente
        t1, t2, b2,    t1, b2, b1,
        # Lateral Trás
        t4, b3, t3,    t4, b4, b3,
        # Lateral Esquerda
        t1, b1, b4,    t1, b4, t4,
        # Lateral Direita
        t2, b3, b2,    t2, t3, b3
    ]
    
    # "Achatando" a lista para o formato que o OpenGL entende
    vertices = []
    for vertice in faces:
        vertices.extend(vertice)
        
    return np.array(vertices, dtype=np.float32)

In [ ]:
# -----------------------------------------------------------------------------
# FUNÇÕES GERADORAS DE FORMAS (Para compor o boneco palito 3D)
# -----------------------------------------------------------------------------
def gerar_cilindro(raio, altura, setores):
    # Gera um cilindro vertical com base em y=0 e topo em y=altura
    vertices = []
    for i in range(setores):
        theta1 = i * 2 * math.pi / setores
        theta2 = (i + 1) * 2 * math.pi / setores
        
        x1, z1 = raio * math.cos(theta1), raio * math.sin(theta1)
        x2, z2 = raio * math.cos(theta2), raio * math.sin(theta2)
        
        # Triângulo 1
        vertices.extend([x1, 0.0, z1,   x2, 0.0, z2,   x1, altura, z1])
        # Triângulo 2
        vertices.extend([x2, 0.0, z2,   x2, altura, z2,   x1, altura, z1])
    return vertices

def gerar_esfera(raio, setores, pilhas):
    # Gera uma esfera centralizada no (0,0,0)
    vertices = []
    for i in range(pilhas):
        lat1 = math.pi * (-0.5 + float(i) / pilhas)
        z1 = math.sin(lat1)
        zr1 = math.cos(lat1)
        
        lat2 = math.pi * (-0.5 + float(i + 1) / pilhas)
        z2 = math.sin(lat2)
        zr2 = math.cos(lat2)
        
        for j in range(setores):
            lng1 = 2 * math.pi * float(j) / setores
            x1, y1 = math.cos(lng1), math.sin(lng1)
            lng2 = 2 * math.pi * float(j + 1) / setores
            x2, y2 = math.cos(lng2), math.sin(lng2)
            
            v1 = [x1 * zr1 * raio, y1 * zr1 * raio, z1 * raio]
            v2 = [x2 * zr1 * raio, y2 * zr1 * raio, z1 * raio]
            v3 = [x1 * zr2 * raio, y1 * zr2 * raio, z2 * raio]
            v4 = [x2 * zr2 * raio, y2 * zr2 * raio, z2 * raio]
            
            vertices.extend(v1 + v2 + v3)
            vertices.extend(v2 + v4 + v3)
    return vertices

def aplicar_transf(vertices, matriz):
    resultado = []
    for i in range(0, len(vertices), 3):
        v = np.array([vertices[i], vertices[i+1], vertices[i+2], 1.0])
        # Com matrizes .T, multiplicamos: vetor x matriz
        v_t = np.dot(v, matriz) 
        resultado.extend([v_t[0], v_t[1], v_t[2]])
    return resultado

# -----------------------------------------------------------------------------
# MONTANDO A NOVA PESSOINHA 3D (Boneco Palito)
# -----------------------------------------------------------------------------
def criar_pessoinha():
    corpo_completo = []
    
    
    cilindro_tronco = gerar_cilindro(raio=0.02, altura=0.4, setores=12)
    
    cilindro_membro = gerar_cilindro(raio=0.015, altura=0.25, setores=12)
    
    esfera_cabeca = gerar_esfera(raio=0.06, setores=16, pilhas=16)
    
    
    mat_tronco = mat_translate(0.0, 0.2, 0.0)
    corpo_completo.extend(aplicar_transf(cilindro_tronco, mat_tronco))
    
    
    mat_cabeca = mat_translate(0.0, 0.68, 0.0)
    corpo_completo.extend(aplicar_transf(esfera_cabeca, mat_cabeca))
    
   
    mat_perna_esq = np.dot(mat_translate(+0.09, -0.2, 0.0), mat_rotate_z(math.pi - 0.4))
    corpo_completo.extend(aplicar_transf(cilindro_membro, mat_perna_esq))
    
    
    mat_perna_dir = np.dot(mat_translate(-0.09, -0.2, 0.0), mat_rotate_z(math.pi + 0.4))
    corpo_completo.extend(aplicar_transf(cilindro_membro, mat_perna_dir))
    
    
    mat_braco_esq = np.dot(mat_translate(+0.23, -0.4, 0.0), mat_rotate_z(math.pi - 0.5))
    corpo_completo.extend(aplicar_transf(cilindro_membro, mat_braco_esq))
    
    
    mat_braco_dir = np.dot(mat_translate(-0.4, +0.18, 0.0), mat_rotate_z(-1.2))
    corpo_completo.extend(aplicar_transf(cilindro_membro, mat_braco_dir))
    
    return np.array(corpo_completo, dtype=np.float32)

In [586]:
def main():
    if not glfw.init():
        return

    window = glfw.create_window(800, 600, "Projeto 1 - Pescaria (SCC0250)", None, None)
    if not window:
        glfw.terminate()
        return

    glfw.make_context_current(window)
    glfw.set_key_callback(window, key_callback)

    # Compila os Shaders
    shader = OpenGL.GL.shaders.compileProgram(
        OpenGL.GL.shaders.compileShader(vertex_shader_source, GL_VERTEX_SHADER),
        OpenGL.GL.shaders.compileShader(fragment_shader_source, GL_FRAGMENT_SHADER)
    )
    glUseProgram(shader)

    # Pega as localizações das variáveis uniformes no shader
    loc_model = glGetUniformLocation(shader, "model")
    loc_color = glGetUniformLocation(shader, "color")

    # Habilita o teste de profundidade para o 3D funcionar corretamente
    glEnable(GL_DEPTH_TEST)

    # Gera os vértices usando nossa função
    vertices_lago = criar_lago()
    
    # Cria os identificadores do VAO e VBO
    vao_lago = glGenVertexArrays(1)
    vbo_lago = glGenBuffers(1)
    
    # Configura o VAO do Lago
    glBindVertexArray(vao_lago)
    glBindBuffer(GL_ARRAY_BUFFER, vbo_lago)
    glBufferData(GL_ARRAY_BUFFER, vertices_lago.nbytes, vertices_lago, GL_STATIC_DRAW)
    
    # Diz ao shader onde os vértices estão (location = 0)
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, 3 * vertices_lago.itemsize, ctypes.c_void_p(0))
    glEnableVertexAttribArray(0)
    
    # Desvincula para não alterar sem querer depois
    glBindBuffer(GL_ARRAY_BUFFER, 0)
    glBindVertexArray(0)

    # Gera os vértices do Barco
    vertices_barco = criar_barco()
    
    vao_barco = glGenVertexArrays(1)
    vbo_barco = glGenBuffers(1)
    
    glBindVertexArray(vao_barco)
    glBindBuffer(GL_ARRAY_BUFFER, vbo_barco)
    glBufferData(GL_ARRAY_BUFFER, vertices_barco.nbytes, vertices_barco, GL_STATIC_DRAW)
    
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, 3 * vertices_barco.itemsize, ctypes.c_void_p(0))
    glEnableVertexAttribArray(0)
    
    # Desvincula
    glBindBuffer(GL_ARRAY_BUFFER, 0)
    glBindVertexArray(0)

    # Gera os vértices da Pessoa
    vertices_pessoa = criar_pessoinha()
    
    vao_pessoa = glGenVertexArrays(1)
    vbo_pessoa = glGenBuffers(1)
    
    glBindVertexArray(vao_pessoa)
    glBindBuffer(GL_ARRAY_BUFFER, vbo_pessoa)
    glBufferData(GL_ARRAY_BUFFER, vertices_pessoa.nbytes, vertices_pessoa, GL_STATIC_DRAW)
    
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, 3 * vertices_pessoa.itemsize, ctypes.c_void_p(0))
    glEnableVertexAttribArray(0)
    
    glBindBuffer(GL_ARRAY_BUFFER, 0)
    glBindVertexArray(0)

    # LOOP PRINCIPAL
    while not glfw.window_should_close(window):
        glfw.poll_events()

        # Limpa a tela com uma cor de céu (azul claro)
        glClearColor(0.5, 0.7, 1.0, 1.0)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        # ---------------------------------------------------------------------
        # AQUI VAMOS DESENHAR OS OBJETOS
        # 1. DESENHANDO O LAGO
        # Cor azul escuro para a água (R, G, B, Alpha)
        cor_lago = np.array([0.0, 0.3, 0.6, 1.0], dtype=np.float32)
        glUniform4fv(loc_color, 1, cor_lago)
        
        # Matriz Identidade (o lago não se move)
        mat_lago = mat_identity()
        glUniformMatrix4fv(loc_model, 1, GL_FALSE, mat_lago)
        
        # Ativa o VAO do lago e desenha os triângulos
        glBindVertexArray(vao_lago)
        numero_de_vertices_lago = len(vertices_lago) // 3
        glDrawArrays(GL_TRIANGLES, 0, numero_de_vertices_lago)  
        
        # --- Dentro do Loop while ---

        # 1. MATRIZ DO BARCO (O barco agora só translada para atender estritamente ao Requisito 16)
        mat_barco_mov = mat_translate(barco_pos_x, barco_pos_y, -1.0)
        # Se quiser manter o barco menor, a escala deve vir ANTES da translação
        mat_barco_esc = mat_scale(0.5, 0.5, 0.5) 
        mat_final_barco = np.dot(mat_barco_esc, mat_barco_mov)

        # Desenha Barco
        glUniform4fv(loc_color, 1, np.array([0.5, 0.25, 0.0, 1.0], dtype=np.float32))
        glUniformMatrix4fv(loc_model, 1, GL_FALSE, mat_final_barco)
        glBindVertexArray(vao_barco)
        glDrawArrays(GL_TRIANGLES, 0, len(vertices_barco)//3)

        # 2. MATRIZ DA PESSOINHA (Filha do Barco)
        # Invertemos a ordem: Primeiro Gira (no eixo próprio), depois Move para o Barco
        mat_pessoinha_rot_local = mat_rotate_y(pessoinha_angulo_y) # Rotaciona no 0,0,0
        mat_pessoinha_pos_local = mat_translate(0.0, 0.2, 0.0)    # Move para o deck

        # Multiplicação correta para Row-Major: Rotaciona -> Posiciona no Barco -> Segue o Barco
        # Assim o Requisito 19 (Rotação via teclado) funcionará no próprio eixo 
        mat_temp = np.dot(mat_pessoinha_rot_local, mat_pessoinha_pos_local)
        mat_final_pessoa = np.dot(mat_temp, mat_final_barco)

        # Desenha Pessoinha
        glUniform4fv(loc_color, 1, np.array([1.0, 0.9, 0.0, 1.0], dtype=np.float32))
        glUniformMatrix4fv(loc_model, 1, GL_FALSE, mat_final_pessoa)
        glBindVertexArray(vao_pessoa)
        glDrawArrays(GL_TRIANGLES, 0, len(vertices_pessoa)//3)
        
        
        glfw.swap_buffers(window)

    glfw.terminate()

if __name__ == "__main__":
    main()